In [ ]:
from pathlib import Path
import xarray as xr
import xesmf as xe
import numpy as np

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root. Start Jupyter from this repository or one of its subdirectories."
    )

REPO_ROOT = find_repo_root(Path.cwd())
RAW_NCAR_DIR = REPO_ROOT / "data" / "raw" / "ncar"
RAW_OISST_DIR = REPO_ROOT / "data" / "raw" / "oisst" / "v2.1"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 1) 파일 경로
# =========================================================
oisst_file = PROCESSED_DIR / "oisst_jja_1991_2023.nc"
uwnd_file  = PROCESSED_DIR / "uwnd_z850_jja_1991_2023.nc"
out_file   = PROCESSED_DIR / "oisst_jja_1991_2023_on_uwnd_grid.nc"

# OISST 변수명 예시: sst, anom 등
oisst_varname = "anom"

# =========================================================
# 2) 보조 함수
# =========================================================
def normalize_latlon(ds):
    rename_dict = {}
    if "latitude" in ds.dims:
        rename_dict["latitude"] = "lat"
    if "longitude" in ds.dims:
        rename_dict["longitude"] = "lon"
    if "latitude" in ds.coords:
        rename_dict["latitude"] = "lat"
    if "longitude" in ds.coords:
        rename_dict["longitude"] = "lon"
    if rename_dict:
        ds = ds.rename(rename_dict)
    return ds

def sort_latlon(ds):
    if "lat" in ds.coords and ds["lat"].values[0] > ds["lat"].values[-1]:
        ds = ds.sortby("lat")
    if "lon" in ds.coords and ds["lon"].values[0] > ds["lon"].values[-1]:
        ds = ds.sortby("lon")
    return ds

def lon_to_0_360(ds):
    lon = ds["lon"]
    ds = ds.assign_coords(lon=xr.where(lon < 0, lon + 360, lon))
    ds = ds.sortby("lon")
    return ds

def lon_to_minus180_180(ds):
    lon = ds["lon"]
    ds = ds.assign_coords(lon=((lon + 180) % 360) - 180)
    ds = ds.sortby("lon")
    return ds

# =========================================================
# 3) 데이터 열기
# =========================================================
ds_oisst = xr.open_dataset(oisst_file)
ds_uwnd  = xr.open_dataset(uwnd_file)

ds_oisst = normalize_latlon(ds_oisst)
ds_uwnd  = normalize_latlon(ds_uwnd)

ds_oisst = sort_latlon(ds_oisst)
ds_uwnd  = sort_latlon(ds_uwnd)

print("OISST data_vars:", list(ds_oisst.data_vars))
print("UWND  data_vars:", list(ds_uwnd.data_vars))

# =========================================================
# 4) 경도 체계 맞추기
#    OISST와 UWND의 lon 범위가 다르면 맞춰줌
# =========================================================
oisst_lon_min = float(ds_oisst.lon.min())
oisst_lon_max = float(ds_oisst.lon.max())
uwnd_lon_min  = float(ds_uwnd.lon.min())
uwnd_lon_max  = float(ds_uwnd.lon.max())

print(f"OISST lon range: {oisst_lon_min:.3f} ~ {oisst_lon_max:.3f}")
print(f"UWND  lon range: {uwnd_lon_min:.3f} ~ {uwnd_lon_max:.3f}")

# target이 0~360이면 source도 0~360으로
if uwnd_lon_min >= 0 and oisst_lon_min < 0:
    ds_oisst = lon_to_0_360(ds_oisst)

# target이 -180~180이면 source도 -180~180으로
elif uwnd_lon_min < 0 and oisst_lon_min >= 0:
    ds_oisst = lon_to_minus180_180(ds_oisst)

ds_oisst = sort_latlon(ds_oisst)
ds_uwnd  = sort_latlon(ds_uwnd)

# =========================================================
# 5) 재격자화에 사용할 source / target grid 준비
# =========================================================
# xESMF는 lat/lon 1D 좌표만 있어도 사용 가능
src = ds_oisst[[oisst_varname]]
dst = ds_uwnd[["lat", "lon"]]

# =========================================================
# 6) regridder 생성
#    bilinear: 가장 일반적
#    reuse_weights=False: 처음 생성
# =========================================================
regridder = xe.Regridder(
    src,
    dst,
    method="bilinear",
    periodic=False,
    reuse_weights=False
)

print(regridder)

# =========================================================
# 7) OISST 변수 재격자화
# =========================================================
oisst_regridded = regridder(ds_oisst[oisst_varname])

# 좌표/이름/속성 정리
oisst_regridded.name = oisst_varname
oisst_regridded.attrs = ds_oisst[oisst_varname].attrs.copy()
oisst_regridded.attrs["regridded_to"] = str(uwnd_file)
oisst_regridded.attrs["regrid_method"] = "xesmf bilinear"

# =========================================================
# 8) Dataset으로 저장
# =========================================================
ds_out = oisst_regridded.to_dataset()

ds_out.to_netcdf(out_file)

print("Saved:", out_file)
print(ds_out)
print("Output shape:", ds_out[oisst_varname].shape)
print("Target lat/lon sizes:", ds_uwnd.lat.size, ds_uwnd.lon.size)